# Drug Exposure — Allscripts Sunrise (SCM)

**OMOP CDM v5.4 — `drug_exposure` table**

### Source Tables
- `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3order` — medication orders
- `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension` — medication details (dose, route, refills)
- `_exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem` — drug formulary with RxNorm codes
- `_exponent._bronze_allscripts_scm_prod_01.dbo_sxammproduct` — product/brand details
- `_exponent._bronze_allscripts_scm_prod_01.dbo_sxammproductpackage` — NDC codes (fallback mapping)
- `_exponent._bronze_allscripts_scm_prod_01.dbo_sxammordertaskoccurrenceadmin` — administration events (Option B)
- `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3ordertaskoccurrence` — task occurrences linking admin → order

### Target
- Silver: `_exponent.omop_silver.drug_exposure`
- Gold: `_exponent.omop.drug_exposure`

### Strategy — Two Options
- **Option A (Order-level)**: One `drug_exposure` per medication order. Grain = `dbo_cv3order.GUID`. Simpler, higher coverage.
- **Option B (Admin-level)**: One `drug_exposure` per administration event. Grain = `OrderTaskOccurrenceAdminID`. More granular, depends on admin table population.

EDA determines which option to use based on admin table coverage.

### Concept Mapping
- Primary path: `RxNormCode` from `dbo_sxammgenericitem` → OMOP `concept` (vocabulary_id = 'RxNorm' / 'RxNorm Extension')
- Fallback path: `NDCCode` from `dbo_sxammproductpackage` → OMOP `concept` (vocabulary_id = 'NDC') → `concept_relationship` 'Maps to' → RxNorm standard concept
- Many RxNorm codes are already `standard_concept = 'S'`; COALESCE handles both cases
- `drug_type_concept_id = 32817` (EHR encounter record)

### Dependencies
- `source_to_person` mapping must be populated for Allscripts SCM patients
- OMOP vocabulary tables (`concept`, `concept_relationship`) must be loaded

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Load Source Data
Load all 7 source tables and print row counts.

In [ ]:
bronze = "_exponent._bronze_allscripts_scm_prod_01"

df_orders  = spark.table(f"{bronze}.dbo_cv3order")
df_medext  = spark.table(f"{bronze}.dbo_cv3medicationextension")
df_gi      = spark.table(f"{bronze}.dbo_sxammgenericitem")
df_prod    = spark.table(f"{bronze}.dbo_sxammproduct")
df_pkg     = spark.table(f"{bronze}.dbo_sxammproductpackage")
df_admin   = spark.table(f"{bronze}.dbo_sxammordertaskoccurrenceadmin")
df_occ     = spark.table(f"{bronze}.dbo_cv3ordertaskoccurrence")

tables = {
    "dbo_cv3order":                     df_orders,
    "dbo_cv3medicationextension":       df_medext,
    "dbo_sxammgenericitem":             df_gi,
    "dbo_sxammproduct":                 df_prod,
    "dbo_sxammproductpackage":          df_pkg,
    "dbo_sxammordertaskoccurrenceadmin": df_admin,
    "dbo_cv3ordertaskoccurrence":       df_occ,
}

for name, df in tables.items():
    print(f"{name}: {df.count():,} rows")

## EDA: Order TypeCode Distribution
Identify which `TypeCode` values represent medication orders. All subsequent filtering depends on this.

In [ ]:
orders_count = df_orders.count()

display(
    df_orders.groupBy("TypeCode")
    .agg(
        F.count("*").alias("count"),
        F.round(F.count("*") / F.lit(orders_count) * 100, 2).alias("pct"),
    )
    .orderBy(F.desc("count"))
)

## EDA: Medication Order Counts & MedExt Join Rate
Filter orders to medication TypeCodes and check join rate to `dbo_cv3medicationextension`.

**Update the `med_type_codes` list below based on Cell 4 results.**

In [ ]:
# UPDATE THIS LIST based on EDA Cell 4 results
med_type_codes = ["MED", "Medication", "Pharm", "IV"]  # <-- adjust after reviewing TypeCode distribution

df_med_orders = df_orders.filter(F.col("TypeCode").isin(med_type_codes))
med_order_count = df_med_orders.count()
print(f"Medication orders: {med_order_count:,}")
print(f"Distinct patients in med orders: {df_med_orders.select('ClientGUID').distinct().count():,}")

# Join to medication extension
df_med_joined = df_med_orders.alias("ord").join(
    df_medext.alias("medext"),
    F.col("ord.GUID") == F.col("medext.GUID"),
    "left",
)

medext_matched = df_med_joined.filter(F.col("medext.GUID").isNotNull()).count()
print(f"\nMed orders with MedExt record: {medext_matched:,} ({medext_matched / med_order_count * 100:.1f}%)")
print(f"Med orders without MedExt record: {med_order_count - medext_matched:,}")

# Check PrescriptionGenericItemID population
gi_id_populated = (
    df_med_joined.filter(
        F.col("medext.PrescriptionGenericItemID").isNotNull()
    ).count()
)
print(f"\nMedExt with PrescriptionGenericItemID: {gi_id_populated:,} ({gi_id_populated / max(medext_matched, 1) * 100:.1f}% of MedExt)")

## EDA: GenericItem Join & RxNorm Coverage
Check whether `dbo_sxammgenericitem` has usable RxNorm codes for concept mapping.

In [ ]:
gi_count = df_gi.count()
print(f"Total generic items: {gi_count:,}")

rxnorm_populated = df_gi.filter(F.col("RxNormCode").isNotNull() & (F.trim(F.col("RxNormCode")) != "")).count()
print(f"Generic items with RxNormCode: {rxnorm_populated:,} ({rxnorm_populated / max(gi_count, 1) * 100:.1f}%)")

# Join med orders → medext → genericitem
df_med_gi = (
    df_med_joined.filter(F.col("medext.PrescriptionGenericItemID").isNotNull())
    .join(
        df_gi.alias("gi"),
        F.col("medext.PrescriptionGenericItemID") == F.col("gi.GenericItemID"),
        "left",
    )
)

gi_matched = df_med_gi.filter(F.col("gi.GenericItemID").isNotNull()).count()
gi_with_rxnorm = df_med_gi.filter(
    F.col("gi.RxNormCode").isNotNull() & (F.trim(F.col("gi.RxNormCode")) != "")
).count()

print(f"\nMed orders matched to GenericItem: {gi_matched:,}")
print(f"Med orders with RxNormCode: {gi_with_rxnorm:,} ({gi_with_rxnorm / max(gi_matched, 1) * 100:.1f}% of matched)")

# Sample GenericItemName + RxNormCode pairs
print("\n=== Sample GenericItemName + RxNormCode (top 20) ===")
display(
    df_gi.filter(F.col("RxNormCode").isNotNull() & (F.trim(F.col("RxNormCode")) != ""))
    .select("GenericItemName", "RxNormCode", "StrengthDesc")
    .distinct()
    .limit(20)
)

## EDA: NDC Code Coverage (Alternative Mapping Path)
Join GenericItem → Product → ProductPackage and check NDC availability as a fallback.

In [ ]:
# Product join
prod_count = df_prod.count()
print(f"Total products: {prod_count:,}")

# ProductPackage with NDC
pkg_count = df_pkg.count()
ndc_populated = df_pkg.filter(
    F.col("NDCCode").isNotNull() & (F.trim(F.col("NDCCode")) != "")
).count()
print(f"Total product packages: {pkg_count:,}")
print(f"Product packages with NDCCode: {ndc_populated:,} ({ndc_populated / max(pkg_count, 1) * 100:.1f}%)")

# Full chain: GenericItem → Product → ProductPackage
df_ndc_chain = (
    df_gi.alias("gi")
    .join(df_prod.alias("prod"), F.col("gi.GenericItemID") == F.col("prod.GenericItemID"), "inner")
    .join(df_pkg.alias("pkg"), F.col("prod.ProductID") == F.col("pkg.ProductID"), "inner")
    .filter(F.col("pkg.NDCCode").isNotNull() & (F.trim(F.col("pkg.NDCCode")) != ""))
)

gi_with_ndc = df_ndc_chain.select("gi.GenericItemID").distinct().count()
print(f"\nGeneric items reachable via NDC path: {gi_with_ndc:,} ({gi_with_ndc / max(gi_count, 1) * 100:.1f}%)")

# Sample NDC codes
print("\n=== Sample NDC codes (top 20) ===")
display(
    df_ndc_chain.select(
        F.col("gi.GenericItemName"),
        F.col("pkg.NDCCode"),
    )
    .distinct()
    .limit(20)
)

## EDA: RxNorm Concept Mapping Preview
Join `RxNormCode` to OMOP `concept` table. Check direct standard concept match rate and `Maps to` resolution.

In [ ]:
df_concept = spark.table("_exponent.omop.concept")

# Distinct RxNorm codes from GenericItem
df_rxnorm_codes = (
    df_gi.filter(F.col("RxNormCode").isNotNull() & (F.trim(F.col("RxNormCode")) != ""))
    .select(F.col("RxNormCode").alias("source_code"))
    .distinct()
)

# Join to concept table
df_rxnorm_mapped = df_rxnorm_codes.join(
    df_concept.alias("c"),
    (F.col("source_code") == F.col("c.concept_code"))
    & (F.col("c.vocabulary_id").isin("RxNorm", "RxNorm Extension")),
    "left",
)

print("=== RxNorm concept match rates ===")
display(
    df_rxnorm_mapped.select(
        F.count("*").alias("total_distinct_rxnorm_codes"),
        F.sum(F.when(F.col("c.concept_id").isNotNull(), 1).otherwise(0)).alias("matched"),
        F.sum(F.when(F.col("c.concept_id").isNull(), 1).otherwise(0)).alias("unmatched"),
        F.round(
            F.sum(F.when(F.col("c.concept_id").isNotNull(), 1).otherwise(0))
            / F.count("*") * 100, 2
        ).alias("match_pct"),
    )
)

# Standard vs non-standard breakdown for matched codes
print("\n=== Standard concept breakdown (matched codes) ===")
display(
    df_rxnorm_mapped.filter(F.col("c.concept_id").isNotNull())
    .groupBy(F.col("c.standard_concept"))
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
)

# For non-standard matched codes, check 'Maps to' resolution
df_concept_rel = spark.table("_exponent.omop.concept_relationship")

df_non_std = df_rxnorm_mapped.filter(
    (F.col("c.concept_id").isNotNull())
    & ((F.col("c.standard_concept") != "S") | F.col("c.standard_concept").isNull())
)

non_std_count = df_non_std.count()
if non_std_count > 0:
    df_maps_to = df_non_std.join(
        df_concept_rel.alias("cr"),
        (F.col("c.concept_id") == F.col("cr.concept_id_1"))
        & (F.col("cr.relationship_id") == "Maps to"),
        "left",
    ).join(
        df_concept.alias("std"),
        (F.col("cr.concept_id_2") == F.col("std.concept_id"))
        & (F.col("std.standard_concept") == "S"),
        "left",
    )
    resolved = df_maps_to.filter(F.col("std.concept_id").isNotNull()).count()
    print(f"\nNon-standard matched codes: {non_std_count:,}")
    print(f"Resolved via 'Maps to': {resolved:,} ({resolved / max(non_std_count, 1) * 100:.1f}%)")

# Sample unmatched codes
print("\n=== Sample unmatched RxNorm codes (top 20) ===")
display(
    df_rxnorm_mapped.filter(F.col("c.concept_id").isNull())
    .select("source_code")
    .limit(20)
)

## EDA: NDC Concept Mapping Preview
Fallback path — join `NDCCode` to OMOP concept table and check `Maps to` standard concept resolution.

In [ ]:
# Distinct NDC codes
df_ndc_codes = (
    df_pkg.filter(F.col("NDCCode").isNotNull() & (F.trim(F.col("NDCCode")) != ""))
    .select(F.col("NDCCode").alias("ndc_code"))
    .distinct()
)

ndc_total = df_ndc_codes.count()

# Join to concept table
df_ndc_mapped = df_ndc_codes.join(
    df_concept.alias("c"),
    (F.col("ndc_code") == F.col("c.concept_code"))
    & (F.col("c.vocabulary_id") == "NDC"),
    "left",
)

ndc_matched = df_ndc_mapped.filter(F.col("c.concept_id").isNotNull()).count()
print(f"Total distinct NDC codes: {ndc_total:,}")
print(f"Matched to concept table: {ndc_matched:,} ({ndc_matched / max(ndc_total, 1) * 100:.1f}%)")

# Check 'Maps to' standard concept resolution
df_ndc_std = (
    df_ndc_mapped.filter(F.col("c.concept_id").isNotNull())
    .join(
        df_concept_rel.alias("cr"),
        (F.col("c.concept_id") == F.col("cr.concept_id_1"))
        & (F.col("cr.relationship_id") == "Maps to"),
        "left",
    )
    .join(
        df_concept.alias("std"),
        (F.col("cr.concept_id_2") == F.col("std.concept_id"))
        & (F.col("std.standard_concept") == "S")
        & (F.col("std.domain_id") == "Drug"),
        "left",
    )
)

ndc_resolved = df_ndc_std.filter(F.col("std.concept_id").isNotNull()).count()
print(f"NDC → 'Maps to' standard Drug concept: {ndc_resolved:,} ({ndc_resolved / max(ndc_matched, 1) * 100:.1f}% of matched)")

## EDA: Date Fields
Determine best fields for `drug_exposure_start_date` and `drug_exposure_end_date`.

- Start: `COALESCE(orders.RequestedDtm, orders.Entered, orders.CreatedWhen)`
- End: `COALESCE(orders.StopDtm, orders.RequestedDtm)`

In [ ]:
df_med_active = df_med_orders.filter(F.col("Active") == True)
med_active_count = df_med_active.count()

date_fields = ["RequestedDtm", "Entered", "StopDtm", "PerformedDtm", "CreatedWhen"]

date_stats = df_med_active.select(
    F.lit(med_active_count).alias("total_med_orders"),
    *[
        expr
        for col_name in date_fields
        for expr in [
            F.count(F.col(col_name)).alias(f"{col_name}_non_null"),
            F.round(
                F.count(F.col(col_name)) / F.lit(med_active_count) * 100, 2
            ).alias(f"{col_name}_pct"),
            F.min(F.col(col_name)).alias(f"{col_name}_min"),
            F.max(F.col(col_name)).alias(f"{col_name}_max"),
        ]
    ]
)

display(date_stats)

# COALESCE start date coverage
coalesce_coverage = df_med_active.select(
    F.count("*").alias("total"),
    F.sum(
        F.when(
            F.coalesce(F.col("RequestedDtm"), F.col("Entered"), F.col("CreatedWhen")).isNotNull(), 1
        ).otherwise(0)
    ).alias("start_date_coalesce_non_null"),
    F.sum(
        F.when(F.col("StopDtm").isNotNull(), 1).otherwise(0)
    ).alias("has_stop_dtm"),
)

print("\n=== COALESCE coverage ===")
display(coalesce_coverage)

## EDA: Dose, Route, Frequency Fields
Distributions for medication detail fields from `dbo_cv3medicationextension`.

In [ ]:
df_medext_active = df_medext.filter(F.col("Active") == True)

print("=== DosageLow — sample values (top 20) ===")
display(
    df_medext_active.groupBy("DosageLow")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .limit(20)
)

print("\n=== Uom distribution (top 20) ===")
display(
    df_medext_active.groupBy("Uom")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .limit(20)
)

print("\n=== OrderRouteCode distribution ===")
display(
    df_medext_active.groupBy("OrderRouteCode")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .limit(20)
)

print("\n=== FormCode distribution (top 20) ===")
display(
    df_medext_active.groupBy("FormCode")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .limit(20)
)

print("\n=== NumRefills distribution ===")
display(
    df_medext_active.groupBy("NumRefills")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .limit(15)
)

# FrequencyCode from orders
print("\n=== FrequencyCode distribution (top 20) ===")
display(
    df_med_orders.groupBy("FrequencyCode")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
    .limit(20)
)

## EDA: Admin Table Population (Option B Viability)
Determine whether administration-level data is populated enough for Option B.

- If admin coverage < 10% of medication orders → Option A only
- If admin coverage > 50% → Option B provides more granular data

In [ ]:
admin_total = df_admin.count()
print(f"Total admin records: {admin_total:,}")

# Filter valid admin events
df_admin_valid = df_admin.filter(
    (F.col("Active") == True)
    & ((F.col("IsMarkedNotDone") == False) | F.col("IsMarkedNotDone").isNull())
)
admin_valid = df_admin_valid.count()
print(f"Valid admin records (Active=T, NotDone=F): {admin_valid:,}")

# Join admin → ordertaskoccurrence → order (medication orders only)
df_admin_to_orders = (
    df_admin_valid.alias("adm")
    .join(
        df_occ.alias("occ"),
        F.col("adm.OrderTaskOccurrenceGUID") == F.col("occ.GUID"),
        "inner",
    )
    .join(
        df_med_orders.alias("ord"),
        F.col("occ.OrderGUID") == F.col("ord.GUID"),
        "inner",
    )
)

admin_med_count = df_admin_to_orders.count()
admin_distinct_orders = df_admin_to_orders.select(F.col("ord.GUID")).distinct().count()

print(f"\nAdmin events linked to medication orders: {admin_med_count:,}")
print(f"Distinct medication orders with admin events: {admin_distinct_orders:,} ({admin_distinct_orders / max(med_order_count, 1) * 100:.1f}% of med orders)")

if admin_distinct_orders / max(med_order_count, 1) < 0.10:
    print("\n>>> LOW COVERAGE — Option A recommended")
elif admin_distinct_orders / max(med_order_count, 1) > 0.50:
    print("\n>>> HIGH COVERAGE — Option B viable for granular admin-level tracking")
else:
    print("\n>>> MODERATE COVERAGE — Consider Option A for completeness, Option B for inpatient meds")

## EDA: Route Concept Mapping
Attempt to map `medext.OrderRouteCode` to OMOP standard Route concepts.

In [ ]:
# Distinct route codes from source
df_routes = (
    df_medext_active.filter(F.col("OrderRouteCode").isNotNull())
    .select(F.col("OrderRouteCode").alias("route_value"))
    .distinct()
)

route_total = df_routes.count()

# Try matching by concept_name (case-insensitive)
df_route_concepts = df_concept.filter(
    (F.col("domain_id") == "Route") & (F.col("standard_concept") == "S")
)

df_route_mapped = df_routes.join(
    df_route_concepts.alias("rc"),
    F.upper(F.col("route_value")) == F.upper(F.col("rc.concept_name")),
    "left",
)

route_matched = df_route_mapped.filter(F.col("rc.concept_id").isNotNull()).count()
print(f"Distinct route codes: {route_total:,}")
print(f"Matched to standard Route concept (by name): {route_matched:,} ({route_matched / max(route_total, 1) * 100:.1f}%)")

# Show matched
print("\n=== Matched routes ===")
display(
    df_route_mapped.filter(F.col("rc.concept_id").isNotNull())
    .select("route_value", F.col("rc.concept_id"), F.col("rc.concept_name"))
    .orderBy("route_value")
)

# Show unmatched
print("\n=== Unmatched routes ===")
display(
    df_route_mapped.filter(F.col("rc.concept_id").isNull())
    .select("route_value")
    .orderBy("route_value")
)

---
# Transformation

## Option A: Order-Level Drug Exposure
One `drug_exposure` per medication order. Grain = `dbo_cv3order.GUID`.

**Update `med_type_codes` IN clause below based on EDA Cell 4 results.**

In [ ]:
source = "allscripts_scm"

# UPDATE THIS based on EDA Cell 4 TypeCode results
med_type_codes_sql = "'MED', 'Medication', 'Pharm', 'IV'"  # <-- adjust

silver_drug_exposure_a = spark.sql(f"""
WITH med_orders AS (
    SELECT
        ord.GUID                          AS order_guid,
        ord.ClientGUID,
        ord.ClientVisitGUID,
        ord.CareProviderGUID,
        ord.Name                          AS order_name,
        ord.SummaryLine,
        ord.RequestedDtm,
        ord.StopDtm,
        ord.Entered                       AS order_entered,
        ord.CreatedWhen                   AS order_created,
        ord.FrequencyCode,
        medext.DosageLow,
        medext.Uom,
        medext.OrderRouteCode,
        medext.FormCode,
        medext.NumRefills,
        medext.RxInstructions,
        medext.DispenseAmount,
        medext.DispenseAmountUnit,
        medext.OrderedAsDisplay,
        gi.RxNormCode,
        gi.GenericItemName
    FROM `{bronze}`.`dbo_cv3order` ord
    INNER JOIN `{bronze}`.`dbo_cv3medicationextension` medext
        ON medext.GUID = ord.GUID
       AND medext.Active = TRUE
    LEFT JOIN `{bronze}`.`dbo_sxammgenericitem` gi
        ON gi.GenericItemID = medext.PrescriptionGenericItemID
       AND gi.Active = TRUE
    WHERE ord.Active = TRUE
      AND ord.TypeCode IN ({med_type_codes_sql})
      AND ord.ClientGUID IS NOT NULL
      AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL
)

SELECT
    -- Drug concept: RxNorm → standard concept (many RxNorm codes are already standard)
    COALESCE(std_concept.concept_id, src_concept.concept_id, 0) AS drug_concept_id,

    -- Start date
    DATE(COALESCE(mo.RequestedDtm, mo.order_entered, mo.order_created)) AS drug_exposure_start_date,
    COALESCE(mo.RequestedDtm, mo.order_entered, mo.order_created) AS drug_exposure_start_datetime,

    -- End date
    DATE(COALESCE(mo.StopDtm, mo.RequestedDtm, mo.order_entered, mo.order_created)) AS drug_exposure_end_date,
    COALESCE(mo.StopDtm, mo.RequestedDtm, mo.order_entered, mo.order_created) AS drug_exposure_end_datetime,

    -- Verbatim end date (only if explicit StopDtm exists)
    CASE WHEN mo.StopDtm IS NOT NULL THEN DATE(mo.StopDtm) ELSE NULL END AS verbatim_end_date,

    -- Type
    32817 AS drug_type_concept_id,

    -- Drug details
    NULL AS stop_reason,
    mo.NumRefills AS refills,
    TRY_CAST(mo.DosageLow AS DOUBLE) AS quantity,
    NULL AS days_supply,
    mo.RxInstructions AS sig,
    COALESCE(route_concept.concept_id, 0) AS route_concept_id,
    mo.OrderRouteCode AS route_source_value,
    mo.Uom AS dose_unit_source_value,
    NULL AS lot_number,

    -- Source values
    COALESCE(mo.OrderedAsDisplay, mo.GenericItemName, mo.order_name) AS drug_source_value,
    COALESCE(src_concept.concept_id, 0) AS drug_source_concept_id,

    -- FK source values
    CONCAT('{source}', ' | ', CAST(mo.ClientGUID AS STRING)) AS person_source_value,
    CASE
        WHEN mo.CareProviderGUID IS NOT NULL
        THEN CONCAT('{source}', ' | ', CAST(mo.CareProviderGUID AS STRING))
        ELSE NULL
    END AS provider_source_value,
    CASE
        WHEN mo.ClientVisitGUID IS NOT NULL
        THEN CONCAT('{source}', ' | ', CAST(mo.ClientVisitGUID AS STRING))
        ELSE NULL
    END AS visit_occurrence_source_value,
    NULL AS visit_detail_source_value,

    -- Unique identifier
    CONCAT('{source}', ' | ', CAST(mo.order_guid AS STRING)) AS drug_exposure_source_value,
    '{source}' AS source_system

FROM med_orders mo

-- RxNorm → source concept
LEFT OUTER JOIN `_exponent`.`omop`.`concept` src_concept
    ON src_concept.concept_code = mo.RxNormCode
   AND src_concept.vocabulary_id IN ('RxNorm', 'RxNorm Extension')

-- Standard concept resolution (for non-standard source concepts)
LEFT OUTER JOIN `_exponent`.`omop`.`concept_relationship` cr
    ON cr.concept_id_1 = src_concept.concept_id
   AND cr.relationship_id = 'Maps to'
LEFT OUTER JOIN `_exponent`.`omop`.`concept` std_concept
    ON std_concept.concept_id = cr.concept_id_2
   AND std_concept.standard_concept = 'S'
   AND std_concept.domain_id = 'Drug'

-- Route concept (best-effort name match — adjust based on EDA Cell 13)
LEFT OUTER JOIN `_exponent`.`omop`.`concept` route_concept
    ON UPPER(route_concept.concept_name) = UPPER(mo.OrderRouteCode)
   AND route_concept.domain_id = 'Route'
   AND route_concept.standard_concept = 'S'

-- Person resolution
INNER JOIN `_exponent`.`omop_mapping`.`source_to_person` stp
    ON stp.person_source_value = CONCAT('{source}', ' | ', CAST(mo.ClientGUID AS STRING))
   AND stp.active_flag = TRUE
""")

print(f"Option A — drug_exposure records: {silver_drug_exposure_a.count():,}")
display(silver_drug_exposure_a)

## Option B: Admin-Level Drug Exposure
One `drug_exposure` per actual administration event. Grain = `OrderTaskOccurrenceAdminID`.

Join chain: admin → ordertaskoccurrence → order → medext → genericitem.

**Only viable if EDA Cell 12 shows sufficient admin table population.**

In [ ]:
silver_drug_exposure_b = spark.sql(f"""
WITH admin_events AS (
    SELECT
        adm.OrderTaskOccurrenceAdminID    AS admin_id,
        occ.ClientGUID,
        occ.OrderGUID,
        adm.AdminDtm,
        adm.AdminQty,
        adm.AdministeredQtyUOMCode,
        adm.AdminByGUID,
        occ.TaskDose,
        occ.TaskUom,
        occ.TaskRouteCode,
        ord.ClientVisitGUID,
        ord.Name                          AS order_name,
        ord.StopDtm,
        medext.OrderRouteCode,
        medext.OrderedAsDisplay,
        medext.RxInstructions,
        gi.RxNormCode,
        gi.GenericItemName
    FROM `{bronze}`.`dbo_sxammordertaskoccurrenceadmin` adm
    INNER JOIN `{bronze}`.`dbo_cv3ordertaskoccurrence` occ
        ON occ.GUID = adm.OrderTaskOccurrenceGUID
       AND occ.Active = TRUE
    INNER JOIN `{bronze}`.`dbo_cv3order` ord
        ON ord.GUID = occ.OrderGUID
       AND ord.Active = TRUE
       AND ord.TypeCode IN ({med_type_codes_sql})
    LEFT JOIN `{bronze}`.`dbo_cv3medicationextension` medext
        ON medext.GUID = ord.GUID
       AND medext.Active = TRUE
    LEFT JOIN `{bronze}`.`dbo_sxammgenericitem` gi
        ON gi.GenericItemID = medext.PrescriptionGenericItemID
       AND gi.Active = TRUE
    WHERE adm.Active = TRUE
      AND (adm.IsMarkedNotDone = FALSE OR adm.IsMarkedNotDone IS NULL)
      AND occ.ClientGUID IS NOT NULL
      AND adm.AdminDtm IS NOT NULL
)

SELECT
    -- Drug concept
    COALESCE(std_concept.concept_id, src_concept.concept_id, 0) AS drug_concept_id,

    -- Dates (admin event = single point in time)
    DATE(ae.AdminDtm) AS drug_exposure_start_date,
    ae.AdminDtm AS drug_exposure_start_datetime,
    DATE(ae.AdminDtm) AS drug_exposure_end_date,
    ae.AdminDtm AS drug_exposure_end_datetime,
    NULL AS verbatim_end_date,

    -- Type
    32817 AS drug_type_concept_id,

    -- Drug details
    NULL AS stop_reason,
    NULL AS refills,
    TRY_CAST(COALESCE(ae.AdminQty, ae.TaskDose) AS DOUBLE) AS quantity,
    NULL AS days_supply,
    ae.RxInstructions AS sig,
    COALESCE(route_concept.concept_id, 0) AS route_concept_id,
    COALESCE(ae.TaskRouteCode, ae.OrderRouteCode) AS route_source_value,
    COALESCE(ae.AdministeredQtyUOMCode, ae.TaskUom) AS dose_unit_source_value,
    NULL AS lot_number,

    -- Source values
    COALESCE(ae.OrderedAsDisplay, ae.GenericItemName, ae.order_name) AS drug_source_value,
    COALESCE(src_concept.concept_id, 0) AS drug_source_concept_id,

    -- FK source values
    CONCAT('{source}', ' | ', CAST(ae.ClientGUID AS STRING)) AS person_source_value,
    CASE
        WHEN ae.AdminByGUID IS NOT NULL
        THEN CONCAT('{source}', ' | ', CAST(ae.AdminByGUID AS STRING))
        ELSE NULL
    END AS provider_source_value,
    CASE
        WHEN ae.ClientVisitGUID IS NOT NULL
        THEN CONCAT('{source}', ' | ', CAST(ae.ClientVisitGUID AS STRING))
        ELSE NULL
    END AS visit_occurrence_source_value,
    NULL AS visit_detail_source_value,

    -- Unique identifier (admin event grain)
    CONCAT('{source}', ' | ', CAST(ae.admin_id AS STRING)) AS drug_exposure_source_value,
    '{source}' AS source_system

FROM admin_events ae

-- RxNorm → source concept
LEFT OUTER JOIN `_exponent`.`omop`.`concept` src_concept
    ON src_concept.concept_code = ae.RxNormCode
   AND src_concept.vocabulary_id IN ('RxNorm', 'RxNorm Extension')

-- Standard concept resolution
LEFT OUTER JOIN `_exponent`.`omop`.`concept_relationship` cr
    ON cr.concept_id_1 = src_concept.concept_id
   AND cr.relationship_id = 'Maps to'
LEFT OUTER JOIN `_exponent`.`omop`.`concept` std_concept
    ON std_concept.concept_id = cr.concept_id_2
   AND std_concept.standard_concept = 'S'
   AND std_concept.domain_id = 'Drug'

-- Route concept
LEFT OUTER JOIN `_exponent`.`omop`.`concept` route_concept
    ON UPPER(route_concept.concept_name) = UPPER(COALESCE(ae.TaskRouteCode, ae.OrderRouteCode))
   AND route_concept.domain_id = 'Route'
   AND route_concept.standard_concept = 'S'

-- Person resolution
INNER JOIN `_exponent`.`omop_mapping`.`source_to_person` stp
    ON stp.person_source_value = CONCAT('{source}', ' | ', CAST(ae.ClientGUID AS STRING))
   AND stp.active_flag = TRUE
""")

print(f"Option B — drug_exposure records: {silver_drug_exposure_b.count():,}")
display(silver_drug_exposure_b)

## Select Option & Create TempView

In [ ]:
# Toggle: set to 'A' or 'B' based on EDA Cell 12 results
OPTION = 'A'

if OPTION == 'A':
    silver_drug_exposure = silver_drug_exposure_a
elif OPTION == 'B':
    silver_drug_exposure = silver_drug_exposure_b
else:
    raise ValueError(f"Invalid OPTION: {OPTION}. Choose 'A' or 'B'.")

print(f"Using Option {OPTION}")
silver_drug_exposure.createOrReplaceTempView("silver_drug_exposure")

## Validation Checks

In [ ]:
total_records = silver_drug_exposure.count()

validation = silver_drug_exposure.select(
    F.lit(total_records).alias("total_drug_exposures"),
    F.countDistinct("person_source_value").alias("distinct_patients"),
    F.sum(
        F.when(F.col("drug_exposure_start_date").isNull(), 1).otherwise(0)
    ).alias("null_start_dates"),
    F.sum(
        F.when(F.col("drug_source_value").isNull(), 1).otherwise(0)
    ).alias("null_drug_source_values"),
    F.sum(
        F.when(F.col("drug_concept_id") == 0, 1).otherwise(0)
    ).alias("unmapped_drug_concept"),
    F.round(
        F.sum(F.when(F.col("drug_concept_id") == 0, 1).otherwise(0))
        / F.lit(total_records) * 100, 2
    ).alias("unmapped_drug_concept_pct"),
    F.sum(
        F.when(F.col("drug_source_concept_id") == 0, 1).otherwise(0)
    ).alias("unmapped_source_concept"),
    F.round(
        F.sum(F.when(F.col("drug_source_concept_id") == 0, 1).otherwise(0))
        / F.lit(total_records) * 100, 2
    ).alias("unmapped_source_concept_pct"),
    F.sum(
        F.when(F.col("person_source_value").isNull(), 1).otherwise(0)
    ).alias("null_person_source_values"),
    F.min("drug_exposure_start_date").alias("min_start_date"),
    F.max("drug_exposure_start_date").alias("max_start_date"),
)

display(validation)

# Quantity statistics
print("\n=== Quantity statistics ===")
display(
    silver_drug_exposure.select(
        F.sum(F.when(F.col("quantity").isNull(), 1).otherwise(0)).alias("quantity_null"),
        F.round(F.min("quantity"), 2).alias("quantity_min"),
        F.round(F.avg("quantity"), 2).alias("quantity_mean"),
        F.round(F.max("quantity"), 2).alias("quantity_max"),
    )
)

# Route concept mapping rate
print("\n=== Route concept mapping rate ===")
display(
    silver_drug_exposure.select(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("route_concept_id") != 0, 1).otherwise(0)).alias("route_mapped"),
        F.round(
            F.sum(F.when(F.col("route_concept_id") != 0, 1).otherwise(0))
            / F.count("*") * 100, 2
        ).alias("route_mapped_pct"),
    )
)

# Top 10 most frequent drug_concept_ids with names
df_concept = spark.table("_exponent.omop.concept")

print("\n=== Top 10 most frequent drug_concept_ids ===")
display(
    silver_drug_exposure.filter(F.col("drug_concept_id") != 0)
    .groupBy("drug_concept_id")
    .agg(F.count("*").alias("count"))
    .join(
        df_concept.select("concept_id", "concept_name"),
        F.col("drug_concept_id") == F.col("concept_id"),
        "left",
    )
    .select("drug_concept_id", "concept_name", "count")
    .orderBy(F.desc("count"))
    .limit(10)
)

---
## Write to Silver Layer

In [ ]:
# -- Merge to Silver layer --
# Uncomment when ready to persist

# spark.sql("""
# MERGE INTO _exponent.omop_silver.drug_exposure AS t
# USING silver_drug_exposure AS s
# ON t.drug_exposure_source_value = s.drug_exposure_source_value
#
# WHEN MATCHED AND (
#      NOT (t.drug_concept_id <=> s.drug_concept_id)
#   OR NOT (t.drug_exposure_start_date <=> s.drug_exposure_start_date)
#   OR NOT (t.drug_exposure_start_datetime <=> s.drug_exposure_start_datetime)
#   OR NOT (t.drug_exposure_end_date <=> s.drug_exposure_end_date)
#   OR NOT (t.drug_exposure_end_datetime <=> s.drug_exposure_end_datetime)
#   OR NOT (t.verbatim_end_date <=> s.verbatim_end_date)
#   OR NOT (t.drug_type_concept_id <=> s.drug_type_concept_id)
#   OR NOT (t.stop_reason <=> s.stop_reason)
#   OR NOT (t.refills <=> s.refills)
#   OR NOT (t.quantity <=> s.quantity)
#   OR NOT (t.days_supply <=> s.days_supply)
#   OR NOT (t.sig <=> s.sig)
#   OR NOT (t.route_concept_id <=> s.route_concept_id)
#   OR NOT (t.lot_number <=> s.lot_number)
#   OR NOT (t.drug_source_value <=> s.drug_source_value)
#   OR NOT (t.drug_source_concept_id <=> s.drug_source_concept_id)
#   OR NOT (t.route_source_value <=> s.route_source_value)
#   OR NOT (t.dose_unit_source_value <=> s.dose_unit_source_value)
#   OR NOT (t.person_source_value <=> s.person_source_value)
#   OR NOT (t.provider_source_value <=> s.provider_source_value)
#   OR NOT (t.visit_occurrence_source_value <=> s.visit_occurrence_source_value)
#   OR NOT (t.source_system <=> s.source_system)
# )
# THEN UPDATE SET
#   t.drug_concept_id               = s.drug_concept_id,
#   t.drug_exposure_start_date      = s.drug_exposure_start_date,
#   t.drug_exposure_start_datetime  = s.drug_exposure_start_datetime,
#   t.drug_exposure_end_date        = s.drug_exposure_end_date,
#   t.drug_exposure_end_datetime    = s.drug_exposure_end_datetime,
#   t.verbatim_end_date             = s.verbatim_end_date,
#   t.drug_type_concept_id          = s.drug_type_concept_id,
#   t.stop_reason                   = s.stop_reason,
#   t.refills                       = s.refills,
#   t.quantity                      = s.quantity,
#   t.days_supply                   = s.days_supply,
#   t.sig                           = s.sig,
#   t.route_concept_id              = s.route_concept_id,
#   t.lot_number                    = s.lot_number,
#   t.drug_source_value             = s.drug_source_value,
#   t.drug_source_concept_id        = s.drug_source_concept_id,
#   t.route_source_value            = s.route_source_value,
#   t.dose_unit_source_value        = s.dose_unit_source_value,
#   t.person_source_value           = s.person_source_value,
#   t.provider_source_value         = s.provider_source_value,
#   t.visit_occurrence_source_value = s.visit_occurrence_source_value,
#   t.visit_detail_source_value     = s.visit_detail_source_value,
#   t.source_system                 = s.source_system,
#   t.last_mod_tsp                  = current_timestamp()
#
# WHEN NOT MATCHED THEN
# INSERT (
#   drug_concept_id,
#   drug_exposure_start_date,
#   drug_exposure_start_datetime,
#   drug_exposure_end_date,
#   drug_exposure_end_datetime,
#   verbatim_end_date,
#   drug_type_concept_id,
#   stop_reason,
#   refills,
#   quantity,
#   days_supply,
#   sig,
#   route_concept_id,
#   lot_number,
#   drug_source_value,
#   drug_source_concept_id,
#   route_source_value,
#   dose_unit_source_value,
#   person_source_value,
#   provider_source_value,
#   visit_occurrence_source_value,
#   visit_detail_source_value,
#   drug_exposure_source_value,
#   source_system,
#   last_mod_tsp
# )
# VALUES (
#   s.drug_concept_id,
#   s.drug_exposure_start_date,
#   s.drug_exposure_start_datetime,
#   s.drug_exposure_end_date,
#   s.drug_exposure_end_datetime,
#   s.verbatim_end_date,
#   s.drug_type_concept_id,
#   s.stop_reason,
#   s.refills,
#   s.quantity,
#   s.days_supply,
#   s.sig,
#   s.route_concept_id,
#   s.lot_number,
#   s.drug_source_value,
#   s.drug_source_concept_id,
#   s.route_source_value,
#   s.dose_unit_source_value,
#   s.person_source_value,
#   s.provider_source_value,
#   s.visit_occurrence_source_value,
#   s.visit_detail_source_value,
#   s.drug_exposure_source_value,
#   s.source_system,
#   current_timestamp()
# );
# """)

## Insert Mapping Records

In [ ]:
# -- Insert new mappings to source_to_drug_exposure --
# Uncomment when ready to persist

# spark.sql("""
# INSERT INTO _exponent.omop_mapping.source_to_drug_exposure (
#     source_system,
#     drug_exposure_source_value,
#     active_flag,
#     created_tsp,
#     last_mod_tsp
# )
# SELECT
#     s.source_system,
#     s.drug_exposure_source_value,
#     TRUE AS active_flag,
#     current_timestamp() AS created_tsp,
#     current_timestamp() AS last_mod_tsp
# FROM (
#     SELECT DISTINCT source_system, drug_exposure_source_value
#     FROM _exponent.omop_silver.drug_exposure
#     WHERE source_system = 'allscripts_scm'
# ) s
# LEFT ANTI JOIN _exponent.omop_mapping.source_to_drug_exposure x
#   ON s.drug_exposure_source_value = x.drug_exposure_source_value;
# """)

## Merge to Gold Layer

In [ ]:
# -- Merge to Gold layer --
# Uncomment when ready to persist

# spark.sql("""
# MERGE INTO _exponent.omop.drug_exposure AS gold
# USING (
#   SELECT
#     sde.drug_exposure_id,
#     stp.person_id,
#     s.drug_concept_id,
#     s.drug_exposure_start_date,
#     s.drug_exposure_start_datetime,
#     s.drug_exposure_end_date,
#     s.drug_exposure_end_datetime,
#     s.verbatim_end_date,
#     s.drug_type_concept_id,
#     s.stop_reason,
#     s.refills,
#     s.quantity,
#     s.days_supply,
#     s.sig,
#     s.route_concept_id,
#     s.lot_number,
#     NULL AS provider_id,
#     NULL AS visit_occurrence_id,
#     NULL AS visit_detail_id,
#     s.drug_source_value,
#     s.drug_source_concept_id,
#     s.route_source_value,
#     s.dose_unit_source_value
#   FROM _exponent.omop_silver.drug_exposure s
#   JOIN _exponent.omop_mapping.source_to_drug_exposure sde
#     ON sde.drug_exposure_source_value = s.drug_exposure_source_value
#    AND sde.active_flag = TRUE
#   JOIN _exponent.omop_mapping.source_to_person stp
#     ON stp.person_source_value = s.person_source_value
#    AND stp.active_flag = TRUE
#   WHERE s.source_system = 'allscripts_scm'
# ) AS src
# ON gold.drug_exposure_id = src.drug_exposure_id
#
# WHEN MATCHED THEN UPDATE SET
#   gold.person_id                    = src.person_id,
#   gold.drug_concept_id              = src.drug_concept_id,
#   gold.drug_exposure_start_date     = src.drug_exposure_start_date,
#   gold.drug_exposure_start_datetime = src.drug_exposure_start_datetime,
#   gold.drug_exposure_end_date       = src.drug_exposure_end_date,
#   gold.drug_exposure_end_datetime   = src.drug_exposure_end_datetime,
#   gold.verbatim_end_date            = src.verbatim_end_date,
#   gold.drug_type_concept_id         = src.drug_type_concept_id,
#   gold.stop_reason                  = src.stop_reason,
#   gold.refills                      = src.refills,
#   gold.quantity                     = src.quantity,
#   gold.days_supply                  = src.days_supply,
#   gold.sig                          = src.sig,
#   gold.route_concept_id             = src.route_concept_id,
#   gold.lot_number                   = src.lot_number,
#   gold.provider_id                  = src.provider_id,
#   gold.visit_occurrence_id          = src.visit_occurrence_id,
#   gold.visit_detail_id              = src.visit_detail_id,
#   gold.drug_source_value            = src.drug_source_value,
#   gold.drug_source_concept_id       = src.drug_source_concept_id,
#   gold.route_source_value           = src.route_source_value,
#   gold.dose_unit_source_value       = src.dose_unit_source_value
#
# WHEN NOT MATCHED THEN INSERT (
#   drug_exposure_id,
#   person_id,
#   drug_concept_id,
#   drug_exposure_start_date,
#   drug_exposure_start_datetime,
#   drug_exposure_end_date,
#   drug_exposure_end_datetime,
#   verbatim_end_date,
#   drug_type_concept_id,
#   stop_reason,
#   refills,
#   quantity,
#   days_supply,
#   sig,
#   route_concept_id,
#   lot_number,
#   provider_id,
#   visit_occurrence_id,
#   visit_detail_id,
#   drug_source_value,
#   drug_source_concept_id,
#   route_source_value,
#   dose_unit_source_value
# )
# VALUES (
#   src.drug_exposure_id,
#   src.person_id,
#   src.drug_concept_id,
#   src.drug_exposure_start_date,
#   src.drug_exposure_start_datetime,
#   src.drug_exposure_end_date,
#   src.drug_exposure_end_datetime,
#   src.verbatim_end_date,
#   src.drug_type_concept_id,
#   src.stop_reason,
#   src.refills,
#   src.quantity,
#   src.days_supply,
#   src.sig,
#   src.route_concept_id,
#   src.lot_number,
#   src.provider_id,
#   src.visit_occurrence_id,
#   src.visit_detail_id,
#   src.drug_source_value,
#   src.drug_source_concept_id,
#   src.route_source_value,
#   src.dose_unit_source_value
# );
# """)